# Civilization ABM — Exploración Interactiva

Este notebook permite explorar el modelo de civilización artificial de forma interactiva.

## Contenido
1. Simulación básica
2. Análisis de métricas
3. Comparación de condiciones
4. Visualización de red social
5. Curva de Lorenz y distribución de riqueza

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from model.model import CivilModel
from analysis.metrics import gini, theil_index, palma_ratio, summary_statistics
from analysis.plots import (
    plot_gini_evolution, plot_wealth_distribution,
    plot_lorenz, plot_network, plot_summary_panel,
    plot_class_evolution, plot_wealth_time_series
)

print('✔ Imports correctos')

## 1. Simulación básica

In [ ]:
# Parámetros
N_AGENTS = 100
STEPS    = 200
SEED     = 42

model = CivilModel(
    N=N_AGENTS,
    initial_inequality=0.8,
    tax_policy='progressive',
    network_type='small_world',
    enforce_floor=False,
    seed=SEED
)

for _ in range(STEPS):
    model.step()

model_df = model.datacollector.get_model_vars_dataframe()
print(f'Simulación completada. Pasos: {len(model_df)}')
model_df.tail()

## 2. Panel de métricas

In [ ]:
fig = plot_summary_panel(model_df, model.schedule.agents, model.network)
plt.show()

In [ ]:
# Métricas finales
stats = summary_statistics(model)
stats.to_frame('valor').style.format('{:.4f}')

## 3. Comparación de políticas fiscales

In [ ]:
policies = ['none', 'flat', 'progressive']
results  = {}

for policy in policies:
    m = CivilModel(
        N=100, initial_inequality=0.8,
        tax_policy=None if policy == 'none' else policy,
        network_type='small_world', seed=42
    )
    for _ in range(200):
        m.step()
    results[policy] = m.datacollector.get_model_vars_dataframe()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
for (policy, df), color in zip(results.items(), colors):
    ax.plot(df.index, df['Gini'], label=f'Política: {policy}',
            color=color, linewidth=2)
ax.set_xlabel('Paso')
ax.set_ylabel('Índice de Gini')
ax.set_title('Efecto de la política fiscal sobre la desigualdad')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Tabla resumen
summary = pd.DataFrame({
    p: {
        'Gini final':   results[p]['Gini'].iloc[-1],
        'Gini medio':   results[p]['Gini'].mean(),
        'Gini min':     results[p]['Gini'].min(),
        'Clase alta (%)': results[p]['UpperClass'].iloc[-1] * 100,
        'Clase baja (%)': results[p]['LowerClass'].iloc[-1] * 100,
    }
    for p in policies
})
summary.style.format('{:.3f}')

## 4. Efecto de la desigualdad inicial

In [ ]:
inequalities = [0.3, 0.8, 1.5]
ineq_results = {}

for ineq in inequalities:
    m = CivilModel(N=100, initial_inequality=ineq,
                   tax_policy='progressive',
                   network_type='small_world', seed=42)
    for _ in range(200):
        m.step()
    ineq_results[ineq] = m.datacollector.get_model_vars_dataframe()

fig, ax = plt.subplots(figsize=(9, 4))
for ineq, df in ineq_results.items():
    ax.plot(df.index, df['Gini'], label=f'σ = {ineq}', linewidth=2)
ax.set_xlabel('Paso')
ax.set_ylabel('Gini')
ax.set_title('Convergencia del Gini según desigualdad inicial')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5. Red social

In [ ]:
if model.network is not None:
    fig = plot_network(model.network, model.schedule.agents, max_nodes=100)
    plt.show()
else:
    print('Red no inicializada en este modelo.')

## 6. Curva de Lorenz final

In [ ]:
wealths = [a.wealth for a in model.schedule.agents]
fig = plot_lorenz(wealths, label=f'Gini = {gini(wealths):.3f}')
plt.show()

## 7. Exportar para paper

In [ ]:
from pathlib import Path

out = Path('../results/notebook')
out.mkdir(parents=True, exist_ok=True)

model_df.to_csv(out / 'model_timeseries.csv')
stats.to_csv(out / 'final_metrics.csv', header=['value'])

print(f'Datos exportados a {out}/')